## Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.ticker import FuncFormatter

# Plot styling
sns.set(style="whitegrid", font_scale=1.2)
sns.set_context("talk")

## Load Matches & Teams

In [2]:
matches = pd.read_parquet("Datasets/SkillCorner Premier League 24-25 data/matches_clean.parquet")

team_lookup = pd.concat([
    matches[["home_team_id","home_team_name"]].rename(columns={"home_team_id":"team_id","home_team_name":"team_name"}),
    matches[["away_team_id","away_team_name"]].rename(columns={"away_team_id":"team_id","away_team_name":"team_name"})
]).drop_duplicates()

## Load Events Data

In [3]:
folder = Path("Datasets/SkillCorner Premier League 24-25 data/dynamic_events_pl_24/dynamic")
dfs = [pd.read_parquet(file) for file in folder.glob("*.parquet")]
events = pd.concat(dfs, ignore_index=True)

print(f"Total events: {len(events)}, Unique matches: {events['match_id'].nunique()}")
print(events["event_type"].value_counts().head(10))

Total events: 1811078, Unique matches: 378
event_type
passing_option        939059
player_possession     362853
on_ball_engagement    326100
off_ball_run          183066
Name: count, dtype: int64


## Load Player Match Data

In [4]:
players_match = pd.read_parquet("Datasets/SkillCorner Premier League 24-25 data/players_match.parquet")

players_match["player_id"] = players_match["id"]
players_match["player_name"] = players_match["short_name"]
players_match["position"] = players_match["player_role_acronym"]
players_match["position_group"] = players_match["player_role_position_group"]

# Minutes summary
minutes_summary = (
    players_match.groupby(["player_id","player_name"])
    .agg(
        minutes_tip=("playing_time_total_minutes_tip","sum"),
        minutes_otip=("playing_time_total_minutes_otip","sum"),
        minutes_played=("playing_time_total_minutes_played","sum")
    )
    .reset_index()
)

# Main position
position_minutes = (
    players_match.groupby(["player_id","player_name","position","position_group"], as_index=False)
    .agg(minutes_played=("playing_time_total_minutes_played","sum"))
)

position_minutes_sorted = position_minutes.sort_values(["player_id","minutes_played"], ascending=[True,False])

def get_main_position(df):
    main_pos = df.iloc[0]["position"]
    pos_group = df.iloc[0]["position_group"]

    if main_pos == "SUB" and len(df) > 1:
        main_pos = df.iloc[1]["position"]
        pos_group = df.iloc[1]["position_group"]

    if main_pos == "GK":
        pos_group = "Goalkeeper"

    return pd.Series({"main_position": main_pos, "position_group": pos_group})

main_positions = position_minutes_sorted.groupby("player_id").apply(get_main_position).reset_index()

# Merge final player dataset
players = minutes_summary.merge(main_positions, on="player_id", how="left")
players["minutes_played"] = players["minutes_played"].replace(0, np.nan)

C:\Users\vicky\AppData\Local\Temp\ipykernel_34884\2970160260.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  main_positions = position_minutes_sorted.groupby("player_id").apply(get_main_position).reset_index()


## Passing Options & Values (xPass × xThreat)

In [5]:
passing_options = events[events["event_type"] == "passing_option"].copy()

passing_options["passer_id"] = passing_options["player_in_possession_id"]
passing_options["passer_name"] = passing_options["player_in_possession_name"]
passing_options["receiver_id"] = passing_options["player_id"]
passing_options["receiver_name"] = passing_options["player_name"]

passing_options["pass_value"] = passing_options["xthreat"] * passing_options["xpass_completion"]

best_option = (
    passing_options.groupby(["match_id","associated_player_possession_event_id"], as_index=False)
    .agg(best_pass_value=("pass_value","max"))
)
passing_options = passing_options.merge(best_option, on=["match_id","associated_player_possession_event_id"], how="left")

# Keep only possessions with >1 passing option
option_counts = passing_options.groupby(["match_id","associated_player_possession_event_id"]).size().reset_index(name="n_options")
passing_options = passing_options.merge(option_counts, on=["match_id","associated_player_possession_event_id"], how="left")
passing_options = passing_options[passing_options["n_options"] > 1]

# Extract actual passes
chosen_passes = passing_options[passing_options["targeted"] == True].copy()

## Decision & Execution Metrics

In [6]:
# Decision quality
chosen_passes["decision_quality"] = chosen_passes["pass_value"] / chosen_passes["best_pass_value"].replace(0,np.nan)
chosen_passes["chose_best"] = chosen_passes["pass_value"] >= chosen_passes["best_pass_value"] * 0.95

# Execution
chosen_passes["actual_completion"] = (chosen_passes["received"] == True).astype(int)
chosen_passes["completion_minus_xpass"] = chosen_passes["actual_completion"] - chosen_passes["xpass_completion"]

## Passer Metrics (Per Player)

In [7]:
passer_metrics = (
    chosen_passes.groupby(["passer_id","passer_name","team_shortname"])
    .agg(
        passes_attempted=("pass_value","count"),
        avg_decision_quality=("decision_quality","mean"),
        chose_best_rate=("chose_best","mean"),
        chose_best_count=("chose_best","sum"),
        chose_not_best_count=("chose_best", lambda x: (~x).sum()),
        completion_minus_xpass_per_pass=("completion_minus_xpass","mean"),
        completion_minus_xpass_total=("completion_minus_xpass","sum"),
        total_pass_value=("pass_value","sum"),
        avg_pass_value=("pass_value","mean"),
        avg_xthreat=("xthreat","mean"),
        avg_xpass_completion=("xpass_completion","mean")
    )
    .reset_index()
).merge(players, left_on="passer_id", right_on="player_id", how="left")

# Per 90 normalisation
passer_metrics["passes_per90_tip"] = passer_metrics["passes_attempted"] / passer_metrics["minutes_tip"] * 90
passer_metrics["pass_value_per90_tip"] = passer_metrics["total_pass_value"] / passer_metrics["minutes_tip"] * 90

# Filter for players with enough minutes
full_games = 5
minutes_threshold = full_games * 90
passers_filtered = passer_metrics[(passer_metrics["passes_attempted"] >= 200) & (passer_metrics["minutes_played"] >= minutes_threshold)]

## Receiver Metrics

In [8]:
receiver_metrics = (
    passing_options.groupby(["receiver_id","receiver_name","team_shortname"])
    .agg(
        passing_options_available=("receiver_id","count"),
        times_targeted=("targeted","sum"),
        option_selection_rate=("targeted","mean"),
        total_option_value=("pass_value","sum"),
        avg_option_value=("pass_value","mean"),
        total_xthreat_available=("xthreat","sum"),
        avg_xthreat_available=("xthreat","mean"),
        avg_xpass_passing_option_created=("xpass_completion","mean")
    )
    .reset_index()
).merge(players, left_on="receiver_id", right_on="player_id", how="left")

receiver_metrics["options_available_per90_tip"] = receiver_metrics["passing_options_available"] / receiver_metrics["minutes_tip"] * 90
receiver_metrics["targets_received_per90_tip"] = receiver_metrics["times_targeted"] / receiver_metrics["minutes_tip"] * 90
receiver_metrics["option_value_per90_tip"] = receiver_metrics["total_option_value"] / receiver_metrics["minutes_tip"] * 90
receiver_metrics["xthreat_available_per90_tip"] = receiver_metrics["total_xthreat_available"] / receiver_metrics["minutes_tip"] * 90

receivers_filtered = receiver_metrics[(receiver_metrics["passing_options_available"] >= 200) & (receiver_metrics["minutes_played"] >= minutes_threshold)]

## Off-Ball Run Metrics (New)

In [9]:
off_ball_runs = events[events["event_type"] == "off_ball_run"].copy()

# Drop columns that are all NaN
off_ball_runs = off_ball_runs.dropna(axis=1, how='all')

pd.set_option('display.max_columns', None)  # ensure all columns are visible
print("Columns remaining:", len(off_ball_runs.columns))
print(off_ball_runs.columns)

Columns remaining: 118
Index(['event_id', 'index', 'match_id', 'frame_start', 'frame_end',
       'time_start', 'time_end', 'minute_start', 'second_start', 'duration',
       ...
       'intended_run_behind', 'push_defensive_line', 'break_defensive_line',
       'passing_option_at_start', 'n_opponents_ahead_end',
       'n_opponents_ahead_start', 'n_opponents_overtaken',
       'is_player_possession_start_matched',
       'is_player_possession_end_matched', 'is_pass_reception_matched'],
      dtype='object', length=118)


In [10]:
off_ball_runs.head(5)

,event_id,index,match_id,frame_start,frame_end,time_start,time_end,minute_start,second_start,duration,period,attacking_side_id,attacking_side,event_type_id,event_type,event_subtype_id,event_subtype,player_id,player_name,player_position_id,player_position,player_in_possession_id,player_in_possession_name,player_in_possession_position_id,player_in_possession_position,team_id,team_shortname,x_start,y_start,channel_id_start,channel_start,third_id_start,third_start,penalty_area_start,x_end,y_end,channel_id_end,channel_end,third_id_end,third_end,penalty_area_end,associated_player_possession_event_id,associated_player_possession_frame_start,game_state_id,game_state,team_score,opponent_team_score,phase_index,team_in_possession_phase_type_id,team_in_possession_phase_type,team_out_of_possession_phase_type_id,team_out_of_possession_phase_type,lead_to_shot,lead_to_goal,distance_covered,trajectory_angle,trajectory_direction_id,trajectory_direction,in_to_out,out_to_in,speed_avg,speed_avg_band_id,speed_avg_band,separation_start,separation_end,separation_gain,last_defensive_line_x_start,last_defensive_line_x_end,delta_to_last_defensive_line_start,delta_to_last_defensive_line_end,delta_to_last_defensive_line_gain,last_defensive_line_height_start,last_defensive_line_height_end,last_defensive_line_height_gain,inside_defensive_shape_start,inside_defensive_shape_end,location_to_player_in_possession_id_start,location_to_player_in_possession_start,location_to_player_in_possession_id_end,location_to_player_in_possession_end,distance_to_player_in_possession_start,distance_to_player_in_possession_end,player_in_possession_x_start,player_in_possession_y_start,player_in_possession_channel_id_start,player_in_possession_channel_start,player_in_possession_third_id_start,player_in_possession_third_start,player_in_possession_penalty_area_start,player_in_possession_x_end,player_in_possession_y_end,player_in_possession_channel_id_end,player_in_possession_channel_end,player_in_possession_third_id_end,player_in_possession_third_end,player_in_possession_penalty_area_end,targeted,received,received_in_space,dangerous,difficult_pass_target,xthreat,xpass_completion,passing_option_score,predicted_passing_option,passing_option_at_player_possession_start,n_simultaneous_runs,give_and_go,intended_run_behind,push_defensive_line,break_defensive_line,passing_option_at_start,n_opponents_ahead_end,n_opponents_ahead_start,n_opponents_overtaken,is_player_possession_start_matched,is_player_possession_end_matched,is_pass_reception_matched
1,1_0,1,1650385,59,72,00:04.9,00:06.2,0,4,1.3,1,2,right_to_left,1,off_ball_run,2.0,coming_short,12941,E. Smith Rowe,19,CF,13068.0,S. Lukić,12.0,LM,48,Fulham,-2.99,8.19,3,center,2,middle_third,False,-6.23,2.77,3,center,2,middle_third,False,8_1,62.0,3,drawing,0,0,0,1,create,9,medium_block,False,False,5.91,-120.87,4.0,sideway_right,False,False,17.30,2.0,running,1.58,1.95,0.37,19.60,19.32,22.59,25.55,2.96,32.90,33.18,0.28,False,True,3.0,ahead,3.0,ahead,10.59,5.41,-9.58,-0.10,3.0,center,2.0,middle_third,False,-9.72,-1.36,3.0,center,2.0,middle_third,False,False,False,None,False,False,0.0013,0.9808,0.7928,True,True,0.0,False,None,None,None,True,9.0,10.0,1.0,True,True,None
15,1_1,15,1650385,125,156,00:11.5,00:14.6,0,11,3.1,1,2,right_to_left,1,off_ball_run,9.0,support,5794,K. Tete,8,RB,12168.0,Adama Traoré,17.0,RW,48,Fulham,-12.74,-29.71,5,wide_right,2,middle_third,False,3.55,-28.08,5,wide_right,2,middle_third,False,8_3,143.0,3,drawing,0,0,0,1,create,9,medium_block,False,False,16.03,5.71,1.0,forward,False,False,19.00,2.0,running,4.65,4.90,0.25,13.68,18.60,26.42,15.05,-11.37,38.82,33.90,-4.92,False,False,1.0,behind,1.0,behind,22.27,7.65,9.35,-32.52,5.0,wide_right,2.0,middle_third,False,10.91,-26.00,5.0,wide_right,2.0,middle_third,False,False,False,None,False,False,0.0020,0.9211,0.9390,True,True,1.0,False,None,None,None,True,6.0,8.0,2.0,True,True,None
20,1_2,20,1650385,145,159,00:13.5,00:14.9,0,13,1.4,1,2,right_to_left,1,off_ball_run,9.0,support,11438,Andreas

In [11]:
# Aggregate off-ball metrics per player
player_offball_metrics = (
    off_ball_runs.groupby(["player_id", "player_name"])
    .agg(
        offball_runs_total=("event_id", "count"),
        offball_distance_total=("distance_covered", "sum"),
        offball_distance_avg=("distance_covered", "mean"),
        offball_speed_avg=("speed_avg", "mean")
    )
    .reset_index()
)

# Merge minutes played from players DataFrame
player_offball_metrics = player_offball_metrics.merge(
    players[['player_id', 'minutes_played']], on='player_id', how='left'
)

## Merge Passing + Off-Ball Metrics

In [12]:
full_player_metrics = passers_filtered.merge(
    player_offball_metrics,
    left_on="passer_id",
    right_on="player_id",
    how="left"
)

In [13]:
full_player_metrics.head()

,passer_id,passer_name,team_shortname,passes_attempted,avg_decision_quality,chose_best_rate,chose_best_count,chose_not_best_count,completion_minus_xpass_per_pass,completion_minus_xpass_total,total_pass_value,avg_pass_value,avg_xthreat,avg_xpass_completion,player_id_x,player_name_x,minutes_tip,minutes_otip,minutes_played_x,main_position,position_group,passes_per90_tip,pass_value_per90_tip,player_id_y,player_name_y,offball_runs_total,offball_distance_total,offball_distance_avg,offball_speed_avg,minutes_played_y
0,82.0,A. Cresswell,West Ham,432,0.606253,0.379630,164,268,0.000583,0.2518,1.267649,0.002934,0.005131,0.858213,82,A. Cresswell,240.62,280.86,937.72,LCB,Central Defender,161.582578,0.474143,82,A. Cresswell,184,2368.31,12.871250,18.010059,937.72
1,124.0,A. Doucouré,Everton,720,0.624121,0.373611,269,451,-0.005271,-3.7950,5.637928,0.007830,0.012702,0.852493,124,A. Doucouré,649.13,886.57,2792.56,AM,Midfield,99.825921,0.781682,124,A. Doucouré,1026,15485.61,15.093187,19.092266,2792.56
2,168.0,A. Armstrong,Southampton,224,0.576774,0.290179,65,159,-0.040264,-9.0192,2.105486,0.009399,0.016215,0.843836,168,A. Armstrong,395.59,356.80,1381.38,CF,Center Forward,50.961854,0.479016,168,A. Armstrong,377,5426.73,14.394509,19.376183,1381.38
3,209.0,A. Smith,Bournemouth,397,0.545485,0.299748,119,278,-0.004091,-1.6242,1.558356,0.003925,0.007419,0.822731,209,A. Smith,436.55,478.56,1753.84,RB,Full Back,81.846295,0.321274,209,A. Smith,313,4998.78,15.970543,19.081797,1753.84
4,494.0,A. Iwobi,Fulham,1299,0.658642,0.382602,497,802,-0.016164,-20.9966,13.324703,0.010258,0.017689,0.787526,494,A. Iwobi,942.91,858.06,3222.12,LW,Wide Attacker,123.988504,1.271832,494,A. Iwobi,1309,18517.29,14.146134,18.942332,3222.12


## On-ball engagement

In [14]:
on_ball_engagement = events[events["event_type"] == "on_ball_engagement"].copy()

# Drop columns that are all NaN
on_ball_engagement = on_ball_engagement.dropna(axis=1, how='all')

pd.set_option('display.max_columns', None)  # ensure all columns are visible
print("Columns remaining:", len(on_ball_engagement.columns))

Columns remaining: 143


In [15]:
on_ball_engagement.head(10)

,event_id,index,match_id,frame_start,frame_end,frame_physical_start,time_start,time_end,minute_start,second_start,duration,period,attacking_side_id,attacking_side,event_type_id,event_type,event_subtype_id,event_subtype,player_id,player_name,player_position_id,player_position,player_in_possession_id,player_in_possession_name,player_in_possession_position_id,player_in_possession_position,team_id,team_shortname,x_start,y_start,channel_id_start,channel_start,third_id_start,third_start,penalty_area_start,x_end,y_end,channel_id_end,channel_end,third_id_end,third_end,penalty_area_end,associated_player_possession_event_id,associated_player_possession_frame_start,associated_player_possession_frame_end,associated_player_possession_end_type_id,associated_player_possession_end_type,game_state_id,game_state,team_score,opponent_team_score,phase_index,team_possession_loss_in_phase,team_in_possession_phase_type_id,team_in_possession_phase_type,team_out_of_possession_phase_type_id,team_out_of_possession_phase_type,game_interruption_before_id,game_interruption_before,game_interruption_after_id,game_interruption_after,lead_to_shot,lead_to_goal,distance_covered,trajectory_angle,trajectory_direction_id,trajectory_direction,speed_avg,speed_avg_band_id,speed_avg_band,last_defensive_line_x_start,last_defensive_line_x_end,delta_to_last_defensive_line_start,delta_to_last_defensive_line_end,delta_to_last_defensive_line_gain,last_defensive_line_height_start,last_defensive_line_height_end,last_defensive_line_height_gain,end_type_id,end_type,consecutive_on_ball_engagements,player_targeted_id,player_targeted_name,player_targeted_position_id,player_targeted_position,player_targeted_distance_to_goal_start,player_targeted_distance_to_goal_end,player_targeted_angle_to_goal_start,player_targeted_angle_to_goal_end,player_targeted_average_speed,player_targeted_speed_avg_band_id,player_targeted_speed_avg_band,speed_difference,n_teammates_ahead_end,n_teammates_ahead_start,n_player_targeted_opponents_ahead_start,n_player_targeted_opponents_ahead_end,n_player_targeted_teammates_ahead_start,n_player_targeted_teammates_ahead_end,n_player_targeted_teammates_within_5m_start,n_player_targeted_teammates_within_5m_end,n_player_targeted_opponents_within_5m_start,n_player_targeted_opponents_within_5m_end,interplayer_distance_start,interplayer_distance_end,interplayer_distance_min,interplayer_distance_start_physical,close_at_player_possession_start,angle_of_engagement,goal_side_start,goal_side_end,pressing_chain,pressing_chain_length,pressing_chain_end_type_id,pressing_chain_end_type,pressing_chain_index,index_in_pressing_chain,simultaneous_defensive_engagement_same_target,simultaneous_defensive_engagement_same_target_rank,affected_line_breaking_passing_option_id,affected_line_break_id,affected_line_break,affected_line_breaking_passing_option_attempted,affected_line_breaking_passing_option_xthreat,affected_line_breaking_passing_option_dangerous,affected_line_breaking_passing_option_run_subtype_id,affected_line_breaking_passing_option_run_subtype,possession_danger,beaten_by_possession,beaten_by_movement,stop_possession_danger,reduce_possession_danger,force_backward,xloss_player_possession_start,xloss_player_possession_end,xloss_player_possession_max,xshot_player_possession_start,xshot_player_possession_end,xshot_player_possession_max,is_player_possession_start_matched,is_player_possession_end_matched,is_previous_pass_matched,is_pass_reception_matched
4,9_0,4,1650385,62,71,59.0,00:04.9,00:06.1,0,4,1.2,1,1,left_to_right,9,on_ball_engagement,11.0,pressing,13078,M. Mount,15,AM,13068.0,S. Lukić,12.0,LM,31,Manchester U,2.56,-6.42,3,center,2,middle_third,False,6.32,-1.46,3,center,2,middle_third,False,8_1,62.0,71.0,1.0,pass,3,drawing,0,0,0,True,1,create,9,medium_block,NaN,None,NaN,None,False,False,7.74,52.84,3.0,sideway_left,22.99,3.0,hsr,-19.61,-19.39,22.17,25.71,3.54,32.89,33.11,0.22,NaN,None,False,13068.0,S. Lukić,12.0,LM,62.10,62.23,-179.76,-178.86,3.40,1.0,jogging,19.59,0.0,0.0,10.

In [16]:
# Define aggregation for on-ball engagements
player_onball_metrics = (
    on_ball_engagement.groupby(["player_id", "player_name"])
    .agg(
        # Basic engagement counts
        obe_total=("event_id", "count"),
        distance_covered_total=("distance_covered", "sum"),
        distance_covered_avg=("distance_covered", "mean"),
        speed_avg_total=("speed_avg", "sum"),
        speed_avg_mean=("speed_avg", "mean"),

        # Impact on shots/goals
        lead_to_shot_total=("lead_to_shot", "sum"),
        lead_to_goal_total=("lead_to_goal", "sum"),

        # Line-breaking passing option metrics
        line_breaks_affected_total=("affected_line_break", lambda x: x.notna().sum()),
        line_breaks_attempted_stopped=("affected_line_breaking_passing_option_attempted", lambda x: x.eq(True).sum()),
        line_breaks_xthreat_total=("affected_line_breaking_passing_option_xthreat", "sum"),
        line_breaks_dangerous_total=("affected_line_breaking_passing_option_dangerous", lambda x: x.eq(True).sum()),

        # Pressing chain metrics
        pressing_chains_total=("pressing_chain", lambda x: x.eq(True).sum()),
        pressing_chains_length_avg=("pressing_chain_length", "mean"),
        pressing_chains_regain=("pressing_chain_end_type", lambda x: (x == "regain").sum()),
        pressing_chains_disruption=("pressing_chain_end_type", lambda x: (x == "disruption").sum()),

        # Defensive effect metrics
        possession_losses_forced=("stop_possession_danger", "sum"),
        beaten_by_possession_total=("beaten_by_possession", "sum"),
        reduce_possession_danger_total=("reduce_possession_danger", "sum")
    )
    .reset_index()
)


player_onball_metrics.head()

,player_id,player_name,obe_total,distance_covered_total,distance_covered_avg,speed_avg_total,speed_avg_mean,lead_to_shot_total,lead_to_goal_total,line_breaks_affected_total,line_breaks_attempted_stopped,line_breaks_xthreat_total,line_breaks_dangerous_total,pressing_chains_total,pressing_chains_length_avg,pressing_chains_regain,pressing_chains_disruption,possession_losses_forced,beaten_by_possession_total,reduce_possession_danger_total
0,82,A. Cresswell,159,1165.16,7.328050,1815.42,12.349796,32,2,3,2,0.0398,1,25,4.400000,4,5,6,1,9
1,124,A. Doucouré,2240,18677.69,8.338254,33903.39,15.244330,153,13,264,105,1.8589,18,928,3.914871,231,55,26,25,21
2,168,A. Armstrong,703,6178.25,8.788407,10419.91,15.101319,42,5,78,30,0.3188,1,315,3.990476,82,22,3,7,4
3,192,A. Lallana,244,1637.97,6.712992,3321.83,13.840958,30,4,17,8,0.1409,1,61,3.901639,11,1,4,4,5
4,209,A. Smith,648,6534.92,10.084753,8819.15,14.110640,79,7,21,15,0.2562,3,153,4.444444,52,15,18,24,14


## Merge On Ball to Main Dataset

In [18]:
# --------------------------
# Step 0: Assume you already have:
# full_player_metrics, player_offball_metrics, player_onball_metrics
# --------------------------

# --------------------------
# Step 1: Clean full_player_metrics
# --------------------------
full_player_metrics_clean = full_player_metrics.rename(columns={
    'player_id_x': 'player_id',
    'player_name_x': 'player_name',
    'minutes_played_x': 'minutes_played',
    'minutes_played_y': 'team_minutes_played'
})

# Keep only relevant columns
columns_to_keep = [
    'player_id', 'player_name', 'main_position', 'position_group',
    'minutes_tip', 'minutes_otip', 'minutes_played',
    'passes_attempted', 'avg_decision_quality', 'chose_best_rate', 
    'chose_best_count', 'chose_not_best_count', 'completion_minus_xpass_per_pass',
    'completion_minus_xpass_total', 'total_pass_value', 'avg_pass_value', 
    'avg_xthreat', 'avg_xpass_completion', 'passes_per90_tip', 'pass_value_per90_tip',
    'team_minutes_played'
]

full_player_metrics_clean = full_player_metrics_clean[columns_to_keep]

# --------------------------
# Step 2: Merge off-ball metrics
# --------------------------
full_player_metrics_clean = full_player_metrics_clean.merge(
    player_offball_metrics[['player_id','offball_runs_total','offball_distance_total','offball_distance_avg','offball_speed_avg']],
    on='player_id',
    how='left'
)

# --------------------------
# Step 3: Merge on-ball metrics
# --------------------------
full_player_metrics_clean = full_player_metrics_clean.merge(
    player_onball_metrics[[
        'player_id',
        'obe_total',
        'distance_covered_total',
        'distance_covered_avg',
        'speed_avg_total',
        'speed_avg_mean',
        'lead_to_shot_total',
        'lead_to_goal_total',
        'line_breaks_affected_total',
        'line_breaks_attempted_stopped',
        'line_breaks_xthreat_total',
        'line_breaks_dangerous_total',
        'pressing_chains_total',
        'pressing_chains_length_avg',
        'pressing_chains_regain',
        'pressing_chains_disruption',
        'possession_losses_forced',
        'beaten_by_possession_total',
        'reduce_possession_danger_total'
    ]],
    on='player_id',
    how='left'
)

# --------------------------
# Step 4: Fill NaNs
# --------------------------
full_player_metrics_clean.fillna(0, inplace=True)

# --------------------------
# Step 5: Reset index
# --------------------------
full_player_metrics_clean.reset_index(drop=True, inplace=True)

C:\Users\vicky\AppData\Local\Temp\ipykernel_34884\2335690159.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  full_player_metrics_clean.fillna(0, inplace=True)


In [19]:
full_player_metrics_clean.head()

,player_id,player_name,main_position,position_group,minutes_tip,minutes_otip,minutes_played,passes_attempted,avg_decision_quality,chose_best_rate,chose_best_count,chose_not_best_count,completion_minus_xpass_per_pass,completion_minus_xpass_total,total_pass_value,avg_pass_value,avg_xthreat,avg_xpass_completion,passes_per90_tip,pass_value_per90_tip,team_minutes_played,offball_runs_total,offball_distance_total,offball_distance_avg,offball_speed_avg,obe_total,distance_covered_total,distance_covered_avg,speed_avg_total,speed_avg_mean,lead_to_shot_total,lead_to_goal_total,line_breaks_affected_total,line_breaks_attempted_stopped,line_breaks_xthreat_total,line_breaks_dangerous_total,pressing_chains_total,pressing_chains_length_avg,pressing_chains_regain,pressing_chains_disruption,possession_losses_forced,beaten_by_possession_total,reduce_possession_danger_total
0,82,A. Cresswell,LCB,Central Defender,240.62,280.86,937.72,432,0.606253,0.379630,164,268,0.000583,0.2518,1.267649,0.002934,0.005131,0.858213,161.582578,0.474143,937.72,184,2368.31,12.871250,18.010059,159.0,1165.16,7.328050,1815.42,12.349796,32.0,2.0,3.0,2.0,0.0398,1.0,25.0,4.400000,4.0,5.0,6,1,9
1,124,A. Doucouré,AM,Midfield,649.13,886.57,2792.56,720,0.624121,0.373611,269,451,-0.005271,-3.7950,5.637928,0.007830,0.012702,0.852493,99.825921,0.781682,2792.56,1026,15485.61,15.093187,19.092266,2240.0,18677.69,8.338254,33903.39,15.244330,153.0,13.0,264.0,105.0,1.8589,18.0,928.0,3.914871,231.0,55.0,26,25,21
2,168,A. Armstrong,CF,Center Forward,395.59,356.80,1381.38,224,0.576774,0.290179,65,159,-0.040264,-9.0192,2.105486,0.009399,0.016215,0.843836,50.961854,0.479016,1381.38,377,5426.73,14.394509,19.376183,703.0,6178.25,8.788407,10419.91,15.101319,42.0,5.0,78.0,30.0,0.3188,1.0,315.0,3.990476,82.0,22.0,3,7,4
3,209,A. Smith,RB,Full Back,436.55,478.56,1753.84,397,0.545485,0.299748,119,278,-0.004091,-1.6242,1.558356,0.003925,0.007419,0.822731,81.846295,0.321274,1753.84,313,4998.78,15.970543,19.081797,648.0,6534.92,10.084753,8819.15,14.110640,79.0,7.0,21.0,15.0,0.2562,3.0,153.0,4.444444,52.0,15.0,18,24,14
4,494,A. Iwobi,LW,Wide Attacker,942.91,858.06,3222.12,1299,0.658642,0.382602,497,802,-0.016164,-20.9966,13.324703,0.010258,0.017689,0.787526,123.988504,1.271832,3222.12,1309,18517.29,14.146134,18.942332,1884.0,15430.97,8.190536,27028.50,14.492493,73.0,10.0,217.0,86.0,1.0815,7.0,833.0,4.069628,241.0,70.0,11,18,13


In [20]:
# --- TIP metrics (offensive, per 90 when team in possession) ---
tip_cols = {
    "passes_attempted": "passes_per90_tip",
    "total_pass_value": "pass_value_per90_tip",
    "offball_runs_total": "offball_runs_per90_tip",
    "offball_distance_total": "offball_distance_per90_tip"
}

for col, per90_col in tip_cols.items():
    full_player_metrics[per90_col] = full_player_metrics[col] / full_player_metrics["minutes_tip"] * 90

# --- OTIP metrics (defensive, per 90 when opponent in possession) ---
otip_cols = {
    "on_ball_engagement_total": "on_ball_engagement_per90_otip",
    "distance_covered_total": "def_distance_covered_per90_otip",
    "speed_avg_total": "def_speed_avg_per90_otip"
}

# Assuming you have aggregated on_ball_engagement similarly to off-ball runs
# Example aggregation:
on_ball_metrics = (
    on_ball_engagement.groupby(["player_id","player_name"])
    .agg(
        on_ball_engagement_total=("event_id","count"),
        distance_covered_total=("distance_covered","sum"),
        speed_avg_total=("speed_avg","mean")
    )
    .reset_index()
)

# Merge with full_player_metrics
full_player_metrics = full_player_metrics.merge(
    on_ball_metrics,
    left_on="passer_id",
    right_on="player_id",
    how="left",
    suffixes=("","_otip")
)

# Compute per90 OTIP metrics
for col, per90_col in otip_cols.items():
    full_player_metrics[per90_col] = full_player_metrics[col] / full_player_metrics["minutes_otip"] * 90

# Optional: fill NaNs with 0 for players who never had that metric
full_player_metrics.fillna(0, inplace=True)

In [21]:
full_player_metrics.head()

,passer_id,passer_name,team_shortname,passes_attempted,avg_decision_quality,chose_best_rate,chose_best_count,chose_not_best_count,completion_minus_xpass_per_pass,completion_minus_xpass_total,total_pass_value,avg_pass_value,avg_xthreat,avg_xpass_completion,player_id_x,player_name_x,minutes_tip,minutes_otip,minutes_played_x,main_position,position_group,passes_per90_tip,pass_value_per90_tip,player_id_y,player_name_y,offball_runs_total,offball_distance_total,offball_distance_avg,offball_speed_avg,minutes_played_y,offball_runs_per90_tip,offball_distance_per90_tip,player_id,player_name,on_ball_engagement_total,distance_covered_total,speed_avg_total,on_ball_engagement_per90_otip,def_distance_covered_per90_otip,def_speed_avg_per90_otip
0,82.0,A. Cresswell,West Ham,432,0.606253,0.379630,164,268,0.000583,0.2518,1.267649,0.002934,0.005131,0.858213,82,A. Cresswell,240.62,280.86,937.72,LCB,Central Defender,161.582578,0.474143,82,A. Cresswell,184,2368.31,12.871250,18.010059,937.72,68.822209,885.827861,82.0,A. Cresswell,159.0,1165.16,12.349796,50.950652,373.368938,3.957422
1,124.0,A. Doucouré,Everton,720,0.624121,0.373611,269,451,-0.005271,-3.7950,5.637928,0.007830,0.012702,0.852493,124,A. Doucouré,649.13,886.57,2792.56,AM,Midfield,99.825921,0.781682,124,A. Doucouré,1026,15485.61,15.093187,19.092266,2792.56,142.251937,2147.035109,124.0,A. Doucouré,2240.0,18677.69,15.244330,227.393212,1896.062465,1.547526
2,168.0,A. Armstrong,Southampton,224,0.576774,0.290179,65,159,-0.040264,-9.0192,2.105486,0.009399,0.016215,0.843836,168,A. Armstrong,395.59,356.80,1381.38,CF,Center Forward,50.961854,0.479016,168,A. Armstrong,377,5426.73,14.394509,19.376183,1381.38,85.770621,1234.626002,168.0,A. Armstrong,703.0,6178.25,15.101319,177.326233,1558.415078,3.809189
3,209.0,A. Smith,Bournemouth,397,0.545485,0.299748,119,278,-0.004091,-1.6242,1.558356,0.003925,0.007419,0.822731,209,A. Smith,436.55,478.56,1753.84,RB,Full Back,81.846295,0.321274,209,A. Smith,313,4998.78,15.970543,19.081797,1753.84,64.528691,1030.558241,209.0,A. Smith,648.0,6534.92,14.110640,121.865597,1228.984453,2.653706
4,494.0,A. Iwobi,Fulham,1299,0.658642,0.382602,497,802,-0.016164,-20.9966,13.324703,0.010258,0.017689,0.787526,494,A. Iwobi,942.91,858.06,3222.12,LW,Wide Attacker,123.988504,1.271832,494,A. Iwobi,1309,18517.29,14.146134,18.942332,3222.12,124.942996,1767.460415,494.0,A. Iwobi,1884.0,15430.97,14.492493,197.608559,1618.520034,1.520085


In [22]:
# List of columns to convert per 90 otip
per90_otip_cols = [
    'obe_total',
    'distance_covered_total',
    'lead_to_shot_total',
    'lead_to_goal_total',
    'line_breaks_affected_total',
    'line_breaks_attempted_stopped',
    'line_breaks_xthreat_total',
    'line_breaks_dangerous_total',
    'pressing_chains_total',
    'pressing_chains_regain',
    'pressing_chains_disruption',
    'possession_losses_forced',
    'beaten_by_possession_total',
    'reduce_possession_danger_total'
]

# Loop through and create per90 columns
for col in per90_otip_cols:
    full_player_metrics[f'{col}_per90_otip'] = full_player_metrics[col] / (full_player_metrics['minutes_otip'] / 90)

KeyError: 'obe_total'